# Claims Analysis Using SDK

This notebook demonstrates how analysts can use the ClaimsAnalyst SDK to analyze stop loss insurance claims data.

## Overview

**Important**: Analysts should ONLY use SDK methods. Do NOT access database or files directly.

The SDK provides high-level, analyst-friendly methods for:
- Loading claims data from database (via SDK)
- Filtering and aggregating claims
- Converting to pandas DataFrames for analysis
- Calculating totals and statistics

All business logic is encapsulated in the SDK - analysts just call simple methods.

## Setup

First, install the package in editable mode:
```bash
pip install -e .
```

Then import and use the SDK:


In [ ]:
from decimal import Decimal
from src.sdk import ClaimsAnalyst

# Initialize the analyst
# SDK handles database connection internally
analyst = ClaimsAnalyst(
    db_path="../warehouse.db",
    approval_threshold=Decimal("100000.00")
)

print("ClaimsAnalyst SDK initialized")
print("Ready to load data from database")


## Load Claims Data

Load claims from the database. The SDK handles all database access and data transformation.

**Note**: Analysts should NOT access the database directly. Use SDK methods only.


In [ ]:
# Load claims from database (silver layer - processed data)
# SDK handles all database access internally
claims = analyst.load_from_database(layer="silver")

print(f"Loaded {len(claims)} claims from database")
print(f"\nFirst claim example:")
if claims:
    first_claim = claims[0]
    print(f"  Claim ID: {first_claim.claim_id}")
    print(f"  Policy ID: {first_claim.policy_id}")
    print(f"  Amount: ${first_claim.claim_amount:,.2f}")
    print(f"  Status: {first_claim.status}")


## Calculate Total Claims

Simple method call - business logic is in the SDK.


In [ ]:
# Calculate total - SDK handles the business logic
total = analyst.get_total_claims(claims)
print(f"Total claims amount: ${total:,.2f}")
if claims:
    print(f"Average claim amount: ${total / len(claims):,.2f}")


## Get Summary Statistics

Get pre-calculated summary statistics. All business logic is in the SDK.


In [ ]:
# Get summary statistics - SDK does all the calculations
stats = analyst.get_summary_statistics(claims)

print("Summary Statistics:")
print(f"  Total Claims: {stats['total_claims']}")
print(f"  Total Amount: ${stats['total_amount']:,.2f}")
print(f"  Average Amount: ${stats['average_amount']:,.2f}")
print(f"\nBy Status:")
for status, count in stats['by_status'].items():
    print(f"  {status}: {count}")
print(f"\nBy Type:")
for claim_type, count in stats['by_type'].items():
    print(f"  {claim_type}: {count}")


## Filter Claims by Status

Simple filter methods - no business logic needed in notebook.


In [ ]:
# Filter claims - SDK methods handle the logic
approved = analyst.get_approved_claims(claims)
paid = analyst.get_paid_claims(claims)
pending = analyst.get_pending_claims(claims)

print(f"Approved claims: {len(approved)}")
print(f"Paid claims: {len(paid)}")
print(f"Pending claims: {len(pending)}")


In [ ]:
# Get high-value claims - threshold logic is in SDK
high_value = analyst.get_high_value_claims(claims, threshold=Decimal("50000.00"))

print(f"High-value claims (>$50k): {len(high_value)}")
if high_value:
    high_value_total = analyst.get_total_claims(high_value)
    print(f"Total high-value claims: ${high_value_total:,.2f}")


## Aggregate Claims by Policy

Group claims by policy - SDK handles the aggregation logic.


In [ ]:
# Aggregate by policy - SDK does the grouping
claims_by_policy = analyst.aggregate_by_policy(claims)

print(f"Claims grouped by {len(claims_by_policy)} policies\n")
for policy_id, policy_claims in list(claims_by_policy.items())[:3]:  # Show first 3
    total = analyst.get_total_claims(policy_claims)
    print(f"Policy {policy_id}:")
    print(f"  Claims: {len(policy_claims)}")
    print(f"  Total: ${total:,.2f}")
    print()
